# 01 — Building the system
> repair input structure → put it in a box → add water → minimize → equilibrate ("settle before you shake")

This is the first of the figure notebooks; see **`00_intro`** for scope and the two-tier setup. Here we take the raw PDB entry and turn it into something an MD engine can actually integrate: a repaired, solvated, energy-minimized system. **Run this notebook first.** It saves the prepared system (`system.xml` + the minimized and equilibrated coordinates) that `02_dynamics` and the others *load* instead of re-preparing, so they don't each redo the (stochastic) preparation.

> **Colab users:** set **Runtime > Change runtime type > T4 GPU** *before* running anything, and set it again for **every** notebook you open. Each notebook runs in its own VM, so the setting does not carry over. Cell 0 prints a warning if it finds no GPU.

> **Terms used in this notebook** (full glossary in [`00_intro`](https://github.com/todd471/MD_tutorial/blob/main/00_intro.md#glossary))
>
> - **PDB entry, model** · The Protein Data Bank file holding the atom coordinates of a solved structure. An NMR entry contains several *models*, alternative coordinate sets that all fit the data; a crystal entry contains one.
> - **Residue notation (ILE4)** · Three-letter amino-acid code plus the residue's position in the sequence: ILE4 is the isoleucine at position 4. Numbering follows the PDB entry (1 to 20 for Trp-cage).
> - **Force field** · The function that assigns a potential energy to any arrangement of the atoms, together with its parameters. The force on each atom is the slope of that function. **CHARMM36** is one widely used protein force field; AMBER ff14SB is another.
> - **Water model** · The force field's description of one water molecule. **TIP3P** (three rigid sites) is used here.
> - **Periodic box** · The simulation cell. Anything that leaves through one face re-enters through the opposite face, so a small box behaves like bulk solution with no walls.
> - **Solvation** · Filling the box around the protein with water molecules, plus ions to neutralize the total charge.
> - **Energy minimization** · Adjusting atomic positions to relieve bad contacts and lower the potential energy before dynamics starts. Not a simulation in time: no velocities, no temperature.
> - **Equilibration** · Short dynamics after minimization and before production, so the packed water and box relax. Here two stages: at fixed volume (NVT) the velocities are drawn and the water loosens; at fixed pressure (NPT) the box resizes to the force field's liquid density. Production then runs at that relaxed box.
> - **System (OpenMM sense)** · The assembled object holding every atom, its force-field parameters, and the box: the thing the simulation advances in time.
> - **Ensemble (two meanings)** · In an NMR entry, the set of models. In simulation, the statistical-mechanics meaning: which bulk quantities are held fixed while the atoms move (see NVT in `02_dynamics`). Each notebook says which meaning it intends.

In [ ]:
#@title Environment on-ramp (imports + modules)
# --- environment on-ramp: make sure the MD stack + the module are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:                          # Colab: provision the stack
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:                                                            # still missing -> almost always the WRONG KERNEL
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
if "google.colab" in sys.modules:                                       # Colab: is this runtime actually a GPU one?
    import openmm as _mm, shutil as _sh                                 # (the pip wheel may expose the T4 as OpenCL, not CUDA)
    _gpu = {"CUDA", "OpenCL"} & {_mm.Platform.getPlatform(i).getName() for i in range(_mm.Platform.getNumPlatforms())}
    if not _gpu and _sh.which("nvidia-smi"):
        print("!" * 78 + "\n!!  A GPU is attached, but OpenMM loaded no CUDA/OpenCL platform (plugin load failures below).\n!!  "
              + str(_mm.Platform.getPluginLoadFailures())[:300] + "\n" + "!" * 78)
    elif not _gpu:
        print("!" * 78 + "\n!!  This Colab runtime has NO GPU. Runtime > Change runtime type > T4 GPU, then\n"
              "!!  Runtime > Run all. (CPU works, but explicit-solvent MD is painfully slow.)\n" + "!" * 78)
    else:
        print(f"Colab GPU runtime OK: OpenMM will use {sorted(_gpu)[0]}.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/todd471/MD_tutorial/main")
for _mod in ("mdtutorial.py", "mdtviz.py", "md_scalogram.py"):          # grab the shipped modules if absent
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import mdtutorial as mdt, mdtviz
PYMOL = mdtviz.setup_pymol()                                            # find/provision headless PyMOL (panels skip if none)

In [ ]:
# --- configuration ---
import numpy as np, mdtraj as md
import matplotlib.pyplot as plt
OUT  = "trpcage_out"   # scratch root: prep, figures, trajectories. Local -- on Colab it's the VM's ephemeral disk.
PREP = OUT             # where 01 SAVES its prepared system and 02/03 LOAD it. LOCAL/standalone: one shared
                       # filesystem, so 02/03 load 01's EXACT prepared system for free. COLAB: each notebook runs
                       # in its own VM with no shared disk, so when 01's prep isn't here 02/03 loudly re-prepare an
                       # INDEPENDENT system (see the load cell) -- fine for dynamics, but a different solvation than
                       # 01's, so NOT bit-identical to it. To share 01's exact prep on Colab, point PREP at a Google
                       # Drive path instead (opt-in; see the README's Colab section).
SEED      = 2024               # SOLVATION/prep seed (repair + water placement). Change it to build a different
                               #   water box; 02+ then load whatever 01 saved.

### 1.1  Fetch the structure and look at the NMR ensemble
An NMR entry is deposited as several models, each consistent with the experimental restraints. The spread between models is a hint at flexibility, not a measurement of it. It reflects both real conformational variety and how tightly the data pin down each region, so a loosely restrained loop looks floppy whether or not it is. We build the simulation from one model, but it is worth seeing the ensemble first. *(Drag to rotate; it cycles through the models on its own.)*

*Code: `fetch_pdb` (in `mdtutorial.py`) downloads the entry from the RCSB; `view_ensemble` (in `mdtviz.py`) animates the deposited models with **py3Dmol**.*

In [ ]:
pdb = mdt.fetch_pdb("1L2Y", OUT)
mdtviz.view_ensemble(pdb).show()

### 1.2  Repair, and why "nothing to fix" is still a real step
For a clean NMR structure PDBFixer finds no missing residues, atoms, or termini. But we do **not** trust the deposited hydrogens. They may not match our force field's residue templates (atom naming) or the protonation state at our chosen pH. So we strip them and rebuild: hydrogens are placed from standard geometric templates with protonation set by the chosen pH, then relaxed against our force field, because we hand that force field to the routine. This is where force-field consistency enters the pipeline. From here on every atom matches the residue templates the `System` will be built from.

*Code: `repair` (in `mdtutorial.py`) runs **PDBFixer** to fix missing atoms and drop heterogens, then calls OpenMM's `Modeller.addHydrogens` **with our force field**, so the new hydrogens are optimized to it rather than to the generic potential PDBFixer's default would use. That step minimizes on a Context, so it is seeded and pinned to the Reference platform for reproducibility (see `determinism`).*

In [ ]:
modeller = mdt.repair(pdb, SEED, OUT)     # strip H, rebuild with the FF (seeded, Reference platform)

### 1.2b  The same repair on a crystal structure — NMR is the *easy* case
Our system is an NMR ensemble, which is unusually clean (NMR resolves hydrogens, so the deposited models already carry them). **The great majority of deposited structures are X-ray crystallographic**, with a small but fast-growing minority from cryo-EM. Both typically need more repair than an NMR model: they rarely resolve hydrogens, so *all* of them are missing and must be built; there are crystallographic waters (and sometimes ions/ligands) to drop; and there are often alternate conformations (altlocs), disordered residues, or nonstandard residues. Running the identical pipeline on ubiquitin (1UBQ, X-ray 1.8 Å; Vijay-Kumar *et al.* 1987) shows the contrast.

And here is what "build the hydrogens" actually looks like. The **stick** view (also called licorice) draws every atom and bond explicitly, and is the representation to reach for when you care about individual atoms rather than overall shape: three residues of ubiquitin as deposited (heavy atoms only, left) vs. after repair (hydrogens in teal, right), Phe ring face-on.

*Code: `repair_report` (in `mdtutorial.py`) runs the same repair path and prints what it changed (missing residues/atoms, termini) for each structure; `hydrogen_sticks` (in `mdtviz.py`) renders the before/after stick views with headless open-source **PyMOL** (skips gracefully if PyMOL isn't installed).*

In [ ]:
#@title §1.2b — same repair on a crystal structure (figure code)
ubq = mdt.fetch_pdb("1UBQ", OUT)
mdt.repair_report(mdt.outp("1L2Y.pdb", OUT), "NMR 1L2Y")
mdt.repair_report(ubq, "X-ray 1UBQ", save=mdt.outp("1UBQ_repaired.pdb", OUT))
import matplotlib.image as mpimg
_vd = os.path.join(OUT, "figures")
before, after = mdtviz.hydrogen_sticks(ubq, mdt.outp("1UBQ_repaired.pdb", OUT), _vd)
if after:
    figH, axH = plt.subplots(1, 2, figsize=(9, 4.5), facecolor="white")
    for a, f, t in zip(axH, (before, after), ("as deposited (X-ray): no hydrogens", "after repair: + hydrogens (teal)")):
        a.imshow(mpimg.imread(f)); a.axis("off"); a.set_title(t, fontsize=11)
    figH.suptitle("Repair up close — 1UBQ residues 4–6 (sticks)", fontsize=12); figH.tight_layout(); plt.show()

### 1.3  Box, water, counter-ions, then build the System, minimize, and equilibrate
We wrap the peptide in a periodic box with about 1 nm of padding, fill it with CHARMM-modified **TIP3P** water (Jorgensen *et al.* 1983; CHARMM LJ-on-H modification, MacKerell *et al.* 1998), and add ions to neutralize the net charge (TC5b is +1 at pH 7, so one Cl⁻ goes in). The peptide goes from a few hundred atoms to a few thousand, almost all of it water. That ratio is the point: in explicit solvent you spend most of your compute on the solvent.

Building the `System` applies the force field to these atoms: every bond, angle, torsion and non-bonded pair gets its parameters, and the result is the object the simulation advances. We use the **CHARMM36** protein force field (Best *et al.* 2012; some OpenMM builds ship the CHARMM36m revision, Huang *et al.* 2017; the exact variant is recorded per run in `run_meta.json`) with **PME** for long-range electrostatics (Darden *et al.* 1993; smooth PME, Essmann *et al.* 1995), a 1 nm cutoff, and `HBonds` constraints so we can use a 2 fs timestep. Energy minimization then adjusts atomic positions to relieve the unfavorable contacts left by packing and lower the potential energy of the system. It is not dynamics: no velocities, no temperature, just a descent to a nearby low-energy arrangement. We record the energy so we can *see* it settle.

**Equilibration** is the last preparation step, and the first that is dynamics. The water was packed into the box by an algorithm at a nominal density, so we let the system relax in two short stages before any production run. First 20 ps at fixed volume (**NVT**): velocities are drawn at 300 K and the water lets go of its packed arrangement. Then 100 ps at fixed pressure (**NPT**): a barostat lets the box resize until the water sits at the force field's own liquid density. Here the box shrinks by about 4 percent. The barostat is then removed, and every production run in `02_dynamics` is NVT at that relaxed box (§2.1b explains why). Then we **save the prepared system** for the other notebooks.

*Code: `solvate` (in `mdtutorial.py`) wraps OpenMM's `Modeller.addSolvent` (TIP3P box + neutralizing ions); `build_system` wraps `ForceField.createSystem` (PME, 1 nm cutoff, `HBonds`); `minimize` wraps `Simulation.minimizeEnergy` (a `LocalEnergyMinimizer`); `equilibrate` runs the NVT stage, then the NPT stage with OpenMM's `MonteCarloBarostat` on a private copy of the System, and stores the relaxed box; `save_prepared` writes `system.xml` + the minimized and equilibrated PDBs that 02 loads.*

In [ ]:
modeller = mdt.solvate(modeller, SEED, OUT)
system, sim, platform, props = mdt.build_system(modeller, SEED)
mdt.print_hardware_report(mdt.hardware_report(sim.context))
min_positions, curve = mdt.minimize(sim, record_curve=True)
prep = mdt.PreparedSystem(sim.topology, system, platform, props, min_positions,
                          mdt.hardware_report(sim.context), modeller=modeller, energy_curve=curve)
mdt.equilibrate(prep, SEED)                               # 20 ps NVT, then 100 ps NPT: the box relaxes to the force field's density
mdt.save_prepared(prep, PREP)                             # PORT: 02+ load this instead of re-prepping (PREP persists to Drive on Colab)
print(f"prepared {system.getNumParticles()} atoms; PE {curve[0]:.0f} -> {curve[-1]:.0f} kJ/mol; saved under {OUT}/")

### Assemble the panels
Panels 1–3 are ray-traced headlessly by open-source PyMOL from the `stage*_*.pdb` snapshots. **Panel 1** shows the repaired fold in the paper's representation: a rainbow **cartoon** (N→C blue→red) with **all atoms as ball-and-stick** under a light translucent grey surface. **Panels 2–3** switch to the **all-atom** peptide (element-colored sticks under the same grey surface) and zoom out to frame the **entire periodic box**, so you can see just how small the solute is relative to the water that fills the cell. Panel 4 is the minimization curve from the live run, and panel 5 is the density of the box during the NPT stage of equilibration: it climbs from the packed value to the force field's liquid density within a few picoseconds and then fluctuates around it. Everything is reproducible from the snapshots (they skip gracefully if PyMOL isn't available).

*Code: `cartoon_panels` (in `mdtviz.py`) ray-traces panels 1–3 with headless open-source **PyMOL** (skipped if it isn't installed); panel 4 replots the energy `curve` that `minimize` recorded back in §1.3, and panel 5 the `density_curve` that `equilibrate` recorded.*

In [ ]:
#@title §1.3 — build-stage cartoon panels (figure code)
panels = mdtviz.cartoon_panels(mdt.outp("stage2_repaired.pdb", OUT),
                               mdt.outp("stage3_solvated.pdb", OUT), os.path.join(OUT, "figures"))
titles = {"panel1_repair": "1. Repair\n(cartoon + atoms + surface)", "panel2_box": "2. Box\n(all-atom, full cell)",
          "panel3_solvated": "3. Add water\n(TIP3P + ions)"}
shown = [(panels[k], titles[k]) for k in ("panel1_repair", "panel2_box", "panel3_solvated") if k in panels]
ncol = len(shown) + 2
fig = plt.figure(figsize=(4.25 * ncol, 5), facecolor="white")
gs = fig.add_gridspec(1, ncol, wspace=0.04)
for i, (f, t) in enumerate(shown):
    ax = fig.add_subplot(gs[0, i]); ax.imshow(mdtviz.sqcrop(mpimg.imread(f))); ax.axis("off"); ax.set_title(t, fontsize=11)
ax = fig.add_subplot(gs[0, len(shown)])
ax.plot(np.array(curve) / 1000, "o-", ms=3, color="firebrick"); ax.set_box_aspect(1.0)
ax.set(xlabel="minimization step", title=f"{len(shown)+1}. Minimize")
ax.set_ylabel("PE (10³ kJ/mol)")                          # /1000 -> narrow labels; put the y-axis on the RIGHT so
ax.yaxis.set_label_position("right"); ax.yaxis.tick_right()   # panel 4's labels don't overlap panel 3 (panels 1-3 unchanged)
ax = fig.add_subplot(gs[0, len(shown) + 1])               # panel 5: the box relaxing under NPT
t_ps, dens = prep.density_curve
ax.plot(t_ps, dens, color="darkorange", lw=1.2); ax.set_box_aspect(1.0)
ax.set(xlabel="NPT time (ps)", title=f"{len(shown)+2}. Equilibrate")
ax.set_ylabel("density (g/cm³)"); ax.yaxis.set_label_position("right"); ax.yaxis.tick_right()
if not shown: print("(cartoon panels skipped — PyMOL unavailable; showing the minimization and density curves only)")
fig.suptitle("Building the system:  repair → box → water → minimize → equilibrate", fontsize=13)
fig.savefig(mdt.outp("figure1.png", OUT), dpi=150, bbox_inches="tight"); plt.show()

### The finished solvated system (interactive)
Protein cartoon (N→C spectrum), faint water-oxygen spheres, green ion(s). Drag to rotate.

*Code: `view_solvated` (in `mdtviz.py`) is a small **py3Dmol** wrapper; waters are drawn as small, light oxygen spheres because py3Dmol's transparency is unreliable, so size and colour carry the faintness rather than true opacity.*

In [ ]:
#@title §1.3 — interactive solvated-system view (code)
mdtviz.view_solvated(mdt.outp("stage3_solvated.pdb", OUT)).show()